### Notebook to create inputs for PRIDICT2.0 where insertion or deletion location is flexible (e.g. stop-codon insertion).

### Description


If edit location is flexible, several PRIDICT2.0 predictions need to be performed to find the highest predicted option. This notebook creates the input files for all possible options (insertions/deletions).

Requirements:
- Target sequence with the possible deletion/insertion region put into square brackets ([ ]; NOT ()!), including **100 bp** context on both sides of the brackets
- PRIDICT2.0 conda environment (includes necessary packages)
- For insertions:
    - Define "insertion" as edit_type
    - Define insert bases (IUPAC base code; includes ATGC, N, R etc.)
    - Define insertion frequency (default = 1; choose e.g. 3 for in-frame insertions)
    - For in-frame insertions: Adjust target sequence context so that ORF starts at the brackets (e.g. add 2bp at the beginning to have 102 wich is dividable by 3)
- For deletions:
    - Define "deletion" as edit_type
    - Define deletion length
    - Define deletion frequency (default = 1; choose e.g. 3 for in-frame deletions)
    - For in-frame deletions: Adjust target sequence context so that ORF starts at the brackets (e.g. add 2bp at the beginning to have 102 wich is dividable by 3)

How to use:
- Use this notebook to create an input batch file for PRIDICT2.0 prediction with flexible mutations (insertions or deletions)

- Input: Target sequence with square brackets ([]) defining region where insertions or deletions options should be created
- Single function: Create flexible mutation inputs for 1 condition
- Batch function: Get the inputs for all conditions in an input .csv file
- Finally run PRIDICT2.0 (batch mode) with created input sequences to get efficiency predictions

Optional:
- Summarize predictions of all flexible mutations options into a single file, by selecting best predicted pegRNA for each variant (last section)

### Functions required to run notebook 
--> Just press run, no changes required.

In [ ]:
# All functions live in flexible_mutation_input.py next to this notebook,
# so that they can also be imported as a module (outside of Jupyter):
#     from flexible_mutation_input import flexible_mutation_sequences
import os
import pandas as pd

from flexible_mutation_input import (
    IUPAC_CODES,
    generate_edits,
    flexible_mutation_sequences,
    handle_duplicate_sequences,
    flexible_mutation_input_generator,
)

### Single mode:

In [ ]:
# Example 1 insertion:
# insert "N" (any base) every 1 bp within the [] brackets
sequence_name = "test_insertion_N"
sequence = "TGCCTGGAGGTGTCTGGGTCCCTCCCCCACCCGACTACTTCACTCTCTGTCCTCTCTGCCCAGGAGCCCAGGATGTGCGAGTTCAAGTGGCTACGGCCGA[CTGTCCTCTCTGCCCAGG]GTGCGAGGCCAGCTCGGGGGCACCGTGGAGCTGCCGTGCCACCTGCTGCCACCTGTTCCTGGACTGTACATCTCCCTGGTGACCTGGCAGCGCCCAGATG"
edit_type = "insertion"
insert = "N"  # Can use IUPAC codes or actual bases (e.g. "N" for any base, "R" for A or G, etc.)
output_dir = "./output"  # folder the csv file is written to
output_file_name = "test_insertion_N_flexible.csv"
step = 1  # Define step size (add the insert every X position within the [] brackets)

df = generate_edits(sequence_name, sequence, edit_type, step, insert_value=insert, output_file_name=output_file_name, output_dir=output_dir, visualise=True)

In [ ]:
# Example 2 insertion:
# set step to 3 and put bracket "[" in-frame to only get in-frame stop codons
sequence_name = "test_insertion_stop"
sequence = "TGCCTGGAGGTGTCTGGGTCCCTCCCCCACCCGACTACTTCACTCTCTGTCCTCTCTGCCCAGGAGCCCAGGATGTGCGAGTTCAAGTGGCTACGGCCGA[CTGTCCTCTCTGCCCAGG]GTGCGAGGCCAGCTCGGGGGCACCGTGGAGCTGCCGTGCCACCTGCTGCCACCTGTTCCTGGACTGTACATCTCCCTGGTGACCTGGCAGCGCCCAGATG"
edit_type = "insertion"
insert = "TAG"  # Can use IUPAC codes
output_dir = "./output"  # folder the csv file is written to
output_file_name = "test_insertion_stop_flexible.csv"
step = 3  # Define step size (add the insert every X position within the [] brackets)

df = generate_edits(sequence_name, sequence, edit_type, step, insert_value=insert, output_file_name=output_file_name, output_dir=output_dir, visualise=True)

In [ ]:
# Example deletion:
# delete 2 bases every 1 bp within the [] brackets
sequence_name = "test_deletion_1"
sequence = "TGCCTGGAGGTGTCTGGGTCCCTCCCCCACCCGACTACTTCACTCTCTGTCCTCTCTGCCCAGGAGCCCAGGATGTGCGAGTTCAAGTGGCTACGGCCGA[CTGTCCTCTCTGCCCAGG]GTGCGAGGCCAGCTCGGGGGCACCGTGGAGCTGCCGTGCCACCTGCTGCCACCTGTTCCTGGACTGTACATCTCCCTGGTGACCTGGCAGCGCCCAGATG"
edit_type = "deletion"
del_length = 2  # Define deletion length (number of bases to delete)
output_dir = "./output"  # folder the csv file is written to
output_file_name = "test_deletion_flexible.csv"
step = 1  # Define step size (add the insert every X position within the [] brackets)

df = generate_edits(sequence_name, sequence, edit_type, step,  del_length=del_length, output_file_name=output_file_name, output_dir=output_dir, visualise=True)

### Batch mode:

In [ ]:
# define input and output paths and filenames
inputpath = './input/'
# check input_flexible_mutations_testfile.csv for details about formatting the input file; 
# #required columns: [Name, sequence_with_brackets, edit_type, insert, deletion_length, step]. insert and deletion_length are optional depending on the edit_type
inputfilename = 'input_flexible_mutations_testfile.csv' 
outputpath = './output/'
summaryoutputfilename = 'summarized_flexible_mutations_outputfile.csv'
#

# run flexible mutation input generator
# (to get the same sequences as an in-memory iterable instead of a csv file, use
#  flexible_mutation_sequences(sequence, edit_type, step=..., insert_value=...) )
inputfiledf, outputfiledf = flexible_mutation_input_generator(inputpath, inputfilename, outputpath, summaryoutputfilename)

# continue with running PRIDICT2 (outside of this notebook) with the 'summarized_flexible_mutations_outputfile.csv' as input file
# example command: python pridict2_pegRNA_design.py batch --input-dir ./addons/flexible_mutations/output --input-fname summarized_flexible_mutations_outputfile.csv --output-dir ./predictions

# Optional but NOT RECOMMENDED: run PRIDICT2 from within this notebook. (uncomment !python command below)
# Caveat: takes a LONG time to run; we recommend running it separately via commandline
# !python ../../pridict2_pegRNA_design.py batch --input-dir ./output --input-fname outputfile.csv --output-dir ../../predictions

### Summarize PRIDICT2.0 predictions of flexible mutations after running PRIDICT2.0

- Only run this after you ran PRIDICT2.0 with the output batch file created above. 

- The code below summarizes all the predictions with different flexible mutations in one file, sorts this by K562 score and saves it as summary prediction file.

- For MMR-deficient context, change "sort_value" from "K562" to "HEK".

- From this summary file, we suggest to take e.g. the top 5 pegRNAs and test these in your experimental setup

In [70]:
### Summarize PRIDICT2 predictions:
pridict2_predictions_folder = '../../predictions/'  # folder with all PRIDICT2 predictions (default "../predictions/")
summary_prediction_output_folder = './summarized_flexible_mutations_pridict2_predictions/' # folder where summarized prediction files will be saved
sort_value = 'K562' # change to "HEK" for sorting to MMR-deficient cell line prediction

# filelist of all .csv files in pridict2_predictions_folder:
filelist = [f for f in os.listdir(pridict2_predictions_folder) if f.endswith('pegRNA_Pridict_full.csv')]

for index, row in inputfiledf.iterrows():
    sequence_name = row['Name']
    print(sequence_name)
    # get all files in filelist that start with the sequence_name
    sequence_files = [f for f in filelist if f.startswith(sequence_name)]
    # read all files and concatenate them
    all_files = []
    for file in sequence_files:
        all_files.append(pd.read_csv(pridict2_predictions_folder+file))
    all_files_df = pd.concat(all_files, ignore_index=True)
    # sort all_files_df by column "PRIDICT2_0_editing_Score_deep_..." (from highest to lowest)
    all_files_df = all_files_df.sort_values(by='PRIDICT2_0_editing_Score_deep_'+sort_value, ascending=False)
    # save concatenated file
    all_files_df.to_csv(summary_prediction_output_folder+sequence_name+'_all_flexible_predictions.csv')

test_insertion_N
test_insertion_stop
test_deletion_1
